In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, STATIC
from src.analysis.pipeline import build_prepared_dataset

TARGET_DATES = ["2026-10-22", "2026-10-23", "2026-10-25", "2026-10-26"]
dataset = build_prepared_dataset(RAW, STATIC, feed="A", target_dates=TARGET_DATES)
print("Liczba wierszy:", len(dataset))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Liczba wierszy: 658876


In [2]:
cross = dataset.groupby(["uncertain_completion", "arrival_delay_extreme"]).size()
print(cross)

uncertain_completion  arrival_delay_extreme
False                 False                    658305
                      True                         89
True                  False                       172
                      True                        310
dtype: int64


In [3]:
print(dataset["arrival_delay_increment"].describe())
print("\nPrzyklad ekstremalnych przyrostow:")
print(dataset[dataset["arrival_delay_increment"].abs() > 3600]
      [["date", "trip_id", "stop_sequence", "arrival_delay", "arrival_delay_increment", "arrival_delay_extreme"]]
      .head(10))

count    625961.00000
mean          0.95997
std          34.78472
min       -2465.00000
25%           0.00000
50%           0.00000
75%           0.00000
max       21013.00000
Name: arrival_delay_increment, dtype: float64

Przyklad ekstremalnych przyrostow:
             date                 trip_id  stop_sequence  arrival_delay  \
565956 2026-10-26  20261024_1_129662021_2           16.0        21241.0   

        arrival_delay_increment  arrival_delay_extreme  
565956                  21013.0                   True  


In [4]:
podejrzane_ale_aktywne = dataset[
    (dataset["arrival_delay_extreme"]) & (~dataset["uncertain_completion"])
]
print(podejrzane_ale_aktywne[["date", "trip_id", "route_id", "arrival_delay"]]
      .sort_values("arrival_delay", ascending=False)
      .head(15))

             date                 trip_id route_id  arrival_delay
565956 2026-10-26  20261024_1_129662021_2      162        21241.0
565957 2026-10-26  20261024_1_129662021_2      162        21219.0
565958 2026-10-26  20261024_1_129662021_2      162        21219.0
565971 2026-10-26  20261024_1_129662022_3      162        20499.0
565972 2026-10-26  20261024_1_129662022_3      162        20499.0
565975 2026-10-26  20261024_1_129662022_3      162        20499.0
565960 2026-10-26  20261024_1_129662022_3      162        20499.0
565973 2026-10-26  20261024_1_129662022_3      162        20499.0
565970 2026-10-26  20261024_1_129662022_3      162        20499.0
565963 2026-10-26  20261024_1_129662022_3      162        20499.0
565964 2026-10-26  20261024_1_129662022_3      162        20499.0
565965 2026-10-26  20261024_1_129662022_3      162        20499.0
565966 2026-10-26  20261024_1_129662022_3      162        20499.0
565967 2026-10-26  20261024_1_129662022_3      162        20499.0
565968 202

In [6]:
powtorzenia = dataset[
    (dataset["trip_id"] == "20261024_1_129662021_2") &
    (dataset["stop_sequence"] == 16)
]
print(powtorzenia[["date", "trip_id", "stop_id", "stop_sequence", "arrival_delay",
                   "observation_count", "last_observed_at"]])

             date                 trip_id stop_id  stop_sequence  \
565956 2026-10-26  20261024_1_129662021_2   10294           16.0   

        arrival_delay  observation_count  last_observed_at  
565956        21241.0                 50        1793012123  


## Wnioski

1. **Oznaczenie `arrival_delay_extreme` (próg 2h)**: oznacza 399 par z nierealnie
   wysokim opóźnieniem, zgodnie z progiem ustalonym w ED.

2. **Zestawienie z oznaczeniem `uncertain_completion` ujawniło trzy odrębne wzorce** stojące za
   ekstremalnymi opóźnieniami:
   - **310 par**: oba oznaczenia jednocześnie, czyli główny, znany już przypadek (linia 146 i podobne):
     opóźnienie realnie rośnie, a ostatnia obserwacja następuje dawno po planowym czasie
     (kurs odwołany lub zakłócony, nieoznaczony w taki sposób w źródle).
   - **172 pary**: `uncertain_completion=True`, ale opóźnienie nie jest ekstremalne, czyli
     kursy-widma ze starym `arrival_time` i zamrożonym, niskim opóźnieniem.
   - **89 par**: opóźnienie ekstremalne, ale `uncertain_completion=False`. ten przypadek zweryfikowano
     na przykładzie (route_id 162, 26.10). To są realne, bardzo poważne, ale aktywnie śledzone
     przez cały czas incydenty (np. wielogodzinny korek/awaria), bez cech porzucenia w
     strumieniu. Prawdopodobnie najbardziej autentyczny z trzech wzorców.

3. **Przyrost opóźnienia (`arrival_delay_increment`) ma akceptowalny rozkład**: mediana i oba
   kwartyle równe 0 (typowy odcinek nie zmienia opóźnienia), średnia 0.96s, z ciężkim ogonem odpowiadającym już
   zidentyfikowanym anomaliom.

4. **Oba oznaczenia (`uncertain_completion`, `arrival_delay_extreme`) oznaczają, nie usuwają.**
   Decyzja o wykluczeniu konkretnych par z konkretnej analizy
   należy do etapu agregacji. Zależnie od tego, czy dany wzorzec zniekształca
   wynik dla tej konkretnej analizy, czy nie.

5. **Powtarzający się łańcuch kroków 1-5 zamknięto w `build_prepared_dataset`**
   Jedno wywołanie zwraca gotowy, w pełni oznaczony zbiór
   analityczny (658 876 wierszy z 4 docelowych dni), gotowy do agregacji.